# Hands-On LangChain — Part 1: Document Loading

This maintained companion notebook demonstrates current LangChain document loaders for local PDF, CSV, Excel, Word, HTML, and Markdown sources. It creates its own small fixtures, so the default path is reproducible and requires no API key.

> YouTube transcription and private Notion exports are documented in the archived article but remain opt-in because they depend on external media, services, or user-owned exports.


## 1. Install dependencies


In [ ]:
!pip install -q "langchain-community>=0.3" pypdf pandas openpyxl unstructured docx2txt python-docx beautifulsoup4


## 2. Create reproducible sample files


In [ ]:
from pathlib import Path
import csv
import pandas as pd
from docx import Document as WordDocument
from pypdf import PdfWriter

DATA_DIR = Path("langchain_document_loading_samples")
DATA_DIR.mkdir(exist_ok=True)

writer = PdfWriter()
writer.add_blank_page(width=612, height=792)
writer.add_blank_page(width=612, height=792)
with (DATA_DIR / "sample.pdf").open("wb") as file:
    writer.write(file)

with (DATA_DIR / "sample.csv").open("w", newline="", encoding="utf-8") as file:
    csv.writer(file).writerows([["topic", "level"], ["RAG", "intro"], ["LangChain", "hands-on"]])

pd.DataFrame({"topic": ["RAG", "LangChain"], "level": ["intro", "hands-on"]}).to_excel(DATA_DIR / "sample.xlsx", index=False)

word = WordDocument()
word.add_heading("Document Loading", level=1)
word.add_paragraph("LangChain converts source files into Document objects.")
word.save(DATA_DIR / "sample.docx")

(DATA_DIR / "sample.html").write_text("<html><body><h1>Document Loading</h1><p>HTML loader example.</p></body></html>")
(DATA_DIR / "sample.md").write_text("# Document Loading\n\nMarkdown loader example.\n")
print(f"Created fixtures in {DATA_DIR.resolve()}")


## 3. Load PDF, CSV, Word, HTML, and Markdown files


In [ ]:
from langchain_community.document_loaders import (
    BSHTMLLoader,
    CSVLoader,
    Docx2txtLoader,
    PyPDFLoader,
    TextLoader,
)

pdf_docs = PyPDFLoader(str(DATA_DIR / "sample.pdf")).load()
csv_docs = CSVLoader(str(DATA_DIR / "sample.csv")).load()
word_docs = Docx2txtLoader(str(DATA_DIR / "sample.docx")).load()
html_docs = BSHTMLLoader(str(DATA_DIR / "sample.html")).load()
markdown_docs = TextLoader(str(DATA_DIR / "sample.md"), encoding="utf-8").load()


## 4. Load Excel rows as LangChain Documents


In [ ]:
from langchain_core.documents import Document

excel_frame = pd.read_excel(DATA_DIR / "sample.xlsx")
excel_docs = [
    Document(
        page_content="\n".join(f"{column}: {row[column]}" for column in excel_frame.columns),
        metadata={"source": str(DATA_DIR / "sample.xlsx"), "row": index},
    )
    for index, row in excel_frame.iterrows()
]


## 5. Verify every loader result


In [ ]:
assert len(pdf_docs) == 2
assert len(csv_docs) == 2
assert len(excel_docs) == 2
assert len(word_docs) == 1 and "LangChain" in word_docs[0].page_content
assert len(html_docs) == 1 and "HTML loader example" in html_docs[0].page_content
assert len(markdown_docs) == 1 and "Markdown loader example" in markdown_docs[0].page_content

print(f"PDF documents: {len(pdf_docs)} pages")
print(f"CSV documents: {len(csv_docs)} rows")
print(f"Excel documents: {len(excel_docs)} rows")
print(f"Word documents: {len(word_docs)} document")
print(f"HTML documents: {len(html_docs)} document")
print(f"Markdown documents: {len(markdown_docs)} document")
print("\nAll local loader assertions passed.")


## 6. Practical production pattern

Map extensions to loader factories, validate each source independently, preserve source metadata, and report failures without discarding successfully loaded documents. Add YouTube or Notion ingestion only when the required external data and credentials are available.
